# Example queries: `applied_helpers` (resstock_oedi)

Auto-generated from `tests/query_snapshots/applied_helpers.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
from buildstock_query.schema.utilities import MappedColumn
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading resstock_2024_amy2018_release_2 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `applied_buildings_all_of_1_2`

get_applied_buildings(all_of=[1, 2]) — buildings where both upgrades 1 and 2 applied successfully. Pins the all_of-only branch SQL: WHERE upgrade IN (1,2) AND applicability GROUP BY <md_keys> HAVING count(distinct(upgrade))=2. Same shape as the prior `applied_in=[1,2]` form.


In [3]:
result = bsq.get_applied_buildings(all_of=[1, 2])
result.head() if hasattr(result, 'head') else result


,bldg_id
0,534814
1,535280
2,1203
3,3674
4,5777


## `applied_buildings_any_of_1_2`

get_applied_buildings(any_of=[1, 2]) — buildings where at least one of upgrades 1 or 2 applied successfully. Pins the any_of-only branch SQL: WHERE upgrade IN (1,2) AND applicability GROUP BY <md_keys> (no HAVING — GROUP BY dedupes).


In [4]:
result = bsq.get_applied_buildings(any_of=[1, 2])
result.head() if hasattr(result, 'head') else result


,bldg_id
0,78
1,87
2,98
3,126
4,244


## `applied_buildings_all_of_1_any_of_2_3`

get_applied_buildings(all_of=[1], any_of=[2, 3]) — buildings where upgrade 1 applied AND at least one of upgrades 2 or 3 also applied. Pins the both-lists branch SQL with the CASE-WHEN HAVING form.


In [5]:
result = bsq.get_applied_buildings(all_of=[1], any_of=[2, 3])
result.head() if hasattr(result, 'head') else result


,bldg_id
0,453398
1,516575
2,130030
3,153320
4,286555


## `query_with_applied_buildings_filter`

Annual baseline restricted via `restrict=[get_applied_buildings_filter(all_of=[1,2]), (state, [CO])]`. Confirms the filter-tuple composes cleanly into the standard restrict pipeline (composite-key tuple-IN on multi-key schemas).


In [6]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    upgrade_id='0',
    restrict=[bsq.get_applied_buildings_filter(all_of=[1, 2]), ('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,metadata_rows_count,model_count,units_count,electricity.total.energy_consumption
0,9071,9071,2.288628e+06,2.194923e+10
